# Lab 03: Clarity Scoring Engine

## Business Context

The analysts' core hypothesis: **companies that can't explain themselves clearly in their S-1 filing tend to underperform.** To test this, we need a scoring engine that rates each section of an S-1 for messaging clarity on a 1-100 scale.

This lab builds that engine using LLM-as-judge, wraps the full agent in the **ChatAgent** interface required for Databricks Model Serving, and registers it in Unity Catalog.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | LLM-as-judge, ChatAgent interface, intent routing, MLflow model registration |
| **Estimated Time** | 35 minutes |
| **Estimated Cost** | ~$2-3 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-llama-4-maverick"

from shared.lab_utils import build_agent
agent, tools, llm = build_agent(include_uc_tools=True, include_scoring=False)
print(f"Agent loaded with {len(tools)} tools: {[t.name for t in tools]}")

## A. Design the Clarity Rubric

An **LLM-as-judge** uses a second LLM to evaluate the output quality of a primary model — or, in our case, to score text quality directly. The rubric defines what each score level means, anchoring the judge's decisions.

The rubric below scores S-1 filing sections on a 1-100 clarity scale. Four section types are scored independently because "clarity" means different things in a business description vs. risk factors.

In [ ]:
CLARITY_RUBRIC = """You are scoring an S-1 filing section for messaging clarity.

## Rubric (1-100 scale)

- **1-20 (Impenetrable):** Dense jargon, circular definitions, no concrete specifics. A reader cannot explain what the company does after reading this section.
- **21-40 (Unclear):** Heavy jargon with occasional concrete details. A domain expert could piece it together, but a general investor would struggle.
- **41-60 (Adequate):** The core message is present but buried in boilerplate. Key details (numbers, timelines, specifics) are sparse.
- **61-80 (Clear):** A general investor can understand the main point. Concrete details are present. Some jargon remains but doesn't obscure meaning.
- **81-100 (Exceptional):** Plain language, specific numbers, clear cause-and-effect. A non-expert could explain this section to someone else accurately.

## Section type: {section_type}

## What to evaluate
- Business Description: What does the company do? For whom? How do they make money?
- Risk Factors: Are the risks specific to this company, or generic boilerplate?
- Competitive Landscape: Does the filing name competitors and explain differentiation?
- Revenue Model: Is the revenue breakdown clear (segments, growth drivers, unit economics)?

## Filing text to score:
{text}

## Instructions
Respond with ONLY a JSON object on a single line:
{{"score": <integer 1-100>, "justification": "<one sentence explaining the score>"}}"""

# Test the rubric manually with a sample passage
from langchain_community.chat_models import ChatDatabricks

judge_llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=200, temperature=0.0)

# Get a sample chunk from Snowflake's filing
sample = spark.sql(f"""
SELECT chunk_text FROM {CATALOG}.{SCHEMA}.filing_chunks
WHERE LOWER(path) LIKE '%snow%'
LIMIT 1
""").first()["chunk_text"]

prompt = CLARITY_RUBRIC.format(section_type="business_description", text=sample[:3000])
result = judge_llm.invoke(prompt)
print("Sample scoring result:")
print(result.content)

## B. Create score_clarity UC Function

Wrapping the scoring logic in a UC function makes it:
- **Callable by the agent** via UCFunctionToolkit
- **Callable from SQL** via `ai_query()` for batch scoring (Lab 06)
- **Governed** by UC permissions
- **Versioned** in the catalog

In [ ]:
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.score_clarity")

spark.sql(f"""
CREATE FUNCTION {CATALOG}.{SCHEMA}.score_clarity(
    filing_text STRING,
    section_type STRING
)
RETURNS STRING
COMMENT 'Score an S-1 filing section for messaging clarity (1-100). Returns JSON with score and justification. Valid section_type values: business_description, risk_factors, competitive_landscape, revenue_model.'
RETURN ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
        'You are scoring an S-1 filing section for messaging clarity on a 1-100 scale.\\n\\n',
        'Rubric:\\n',
        '1-20 (Impenetrable): Dense jargon, circular definitions, no specifics.\\n',
        '21-40 (Unclear): Heavy jargon with occasional concrete details.\\n',
        '41-60 (Adequate): Core message present but buried in boilerplate.\\n',
        '61-80 (Clear): General investor can understand. Concrete details present.\\n',
        '81-100 (Exceptional): Plain language, specific numbers, clear cause-and-effect.\\n\\n',
        'Section type: ', section_type, '\\n\\n',
        'Filing text:\\n', SUBSTRING(filing_text, 1, 6000), '\\n\\n',
        'Respond with ONLY a JSON object: {{\"score\": <int>, \"justification\": \"<one sentence>\"}}'
    )
)::STRING
""")

print(f"Created: {CATALOG}.{SCHEMA}.score_clarity")

# Test it
display(spark.sql(f"""
SELECT {CATALOG}.{SCHEMA}.score_clarity(
    (SELECT chunk_text FROM {CATALOG}.{SCHEMA}.filing_chunks WHERE LOWER(path) LIKE '%snow%' LIMIT 1),
    'business_description'
) AS clarity_score
"""))

## C. Intent Routing

Instead of letting the agent guess which tool to use, we classify the user's intent upfront and route to the appropriate tool subset. This reduces unnecessary tool calls and improves response quality.

| Intent | Description | Primary Tools |
|---|---|---|
| `RESEARCH` | Content or concept questions about filings | search_filings |
| `STOCK_LOOKUP` | Stock performance questions | get_stock_performance |
| `CLARITY_SCORE` | Score a section for clarity | score_clarity |
| `COMPARISON` | Cross-data analysis | search_filings + get_stock_performance |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

classifier_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an intent classifier for an IPO filing analyzer.\n"
     "Classify the user's query into exactly one of: RESEARCH, STOCK_LOOKUP, CLARITY_SCORE, COMPARISON.\n\n"
     "RESEARCH — questions about filing content (risk factors, business model, competition)\n"
     "STOCK_LOOKUP — questions about stock price or returns\n"
     "CLARITY_SCORE — requests to score or evaluate filing clarity\n"
     "COMPARISON — questions combining filing analysis with stock performance\n\n"
     "Respond with only the label."),
    ("human", "{query}"),
])

classifier_chain = classifier_prompt | llm | StrOutputParser()

# Test
test_queries = [
    ("RESEARCH",      "What are Snowflake's key risk factors?"),
    ("STOCK_LOOKUP",  "How did COIN perform in its first year?"),
    ("CLARITY_SCORE", "Score Coinbase's risk factors for clarity"),
    ("COMPARISON",    "Do companies with clearer business descriptions perform better?"),
]

for expected, query in test_queries:
    intent = classifier_chain.invoke({"query": query}).strip().upper()
    match = "OK" if intent == expected else "MISMATCH"
    print(f"  [{match}] '{query}' -> {intent} (expected {expected})")

## D. ChatAgent + UC Registration

Databricks Model Serving requires models to implement the **ChatAgent** interface. This ensures:
- Consistent request/response schema (OpenAI-compatible message format)
- Automatic tracing integration with MLflow
- Compatibility with the AI Gateway and Review App

`ChatAgent.predict(messages, context)` is the single required method.

In [ ]:
import uuid
try:
    from databricks.agents import ChatAgent, ChatAgentMessage, ChatAgentResponse
except ImportError:
    from mlflow.pyfunc import ChatAgent
    from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse
from typing import Optional

# Reload tools with scoring enabled
from shared.lab_utils import build_agent as _build
_, all_tools_with_scoring, _ = _build(include_uc_tools=True, include_scoring=True)

INTENT_PROMPTS = {
    "RESEARCH": (
        "You are an IPO Filing Analyzer. Use search_filings to find relevant passages, "
        "then synthesize a grounded answer with citations."
    ),
    "STOCK_LOOKUP": (
        "You are a stock data assistant. Use get_stock_performance to look up the requested ticker. "
        "Report the numbers clearly. Do NOT give investment advice."
    ),
    "CLARITY_SCORE": (
        "You are a filing clarity scorer. Use search_filings to find the relevant section, "
        "then use score_clarity to score it. Report the score and justification."
    ),
    "COMPARISON": (
        "You are an IPO analyst. Use search_filings and get_stock_performance to gather data, "
        "then compare and analyze. Ground every claim in the data. Do NOT give investment advice."
    ),
}


class IpoAnalyzerAgent(ChatAgent):
    """ChatAgent wrapping the IPO Filing Analyzer with intent routing."""

    def __init__(self):
        from langchain_community.chat_models import ChatDatabricks
        from langgraph.prebuilt import create_react_agent
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser

        self._llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)
        self._tools = all_tools_with_scoring
        self._classifier = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Classify into: RESEARCH, STOCK_LOOKUP, CLARITY_SCORE, COMPARISON.\n"
                 "Respond with only the label."),
                ("human", "{query}"),
            ]) | self._llm | StrOutputParser()
        )

    def predict(self, messages: list[ChatAgentMessage], context: Optional[dict] = None) -> ChatAgentResponse:
        from langgraph.prebuilt import create_react_agent

        user_query = messages[-1].content if messages else ""

        intent = self._classifier.invoke({"query": user_query}).strip().upper()
        if intent not in INTENT_PROMPTS:
            intent = "RESEARCH"

        routed_agent = create_react_agent(
            self._llm, self._tools, prompt=INTENT_PROMPTS[intent]
        )
        result = routed_agent.invoke(
            {"messages": [{"role": "user", "content": user_query}]}
        )

        return ChatAgentResponse(
            messages=[
                ChatAgentMessage(
                    role="assistant",
                    id=str(uuid.uuid4()),
                    content=result["messages"][-1].content,
                )
            ]
        )


# Smoke test
agent_obj = IpoAnalyzerAgent()
test_msg = ChatAgentMessage(role="user", id=str(uuid.uuid4()), content="Score Snowflake's business description for clarity.")
response = agent_obj.predict([test_msg])
print("ChatAgent response:")
print(response.messages[0].content)

In [ ]:
import mlflow

username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/ipo-analyzer/lab-03-clarity-engine")

# ChatAgent auto-infers its model signature — do not pass signature= explicitly
with mlflow.start_run(run_name="ipo-analyzer-v1") as run:
    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "rubric_version": "v1",
        "intent_classes": "RESEARCH,STOCK_LOOKUP,CLARITY_SCORE,COMPARISON",
    })

    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=IpoAnalyzerAgent(),
        pip_requirements=[
            "databricks-sdk", "databricks-vectorsearch", "mlflow",
            "langchain", "langchain-community", "langgraph",
            "unitycatalog-langchain", "databricks-agents",
        ],
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")

# Register in Unity Catalog
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.ipo_filing_agent"

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=UC_MODEL_NAME,
)

print(f"\nRegistered: {registered.name} v{registered.version}")

## Before / After

**Before:** The agent can answer questions and look up stock data, but it can't evaluate *how clearly* a company communicates. Asking "Score Coinbase's risk factors for clarity" gets a confused response.

**After:** The agent scores any S-1 section on a 1-100 clarity scale with a justification. It's wrapped in ChatAgent and registered in Unity Catalog — ready for deployment.

In [ ]:
# The clarity scoring question
print("=== CLARITY SCORING ===")
test = agent_obj.predict([
    ChatAgentMessage(role="user", id=str(uuid.uuid4()),
                     content="Score Coinbase's risk factors section for clarity.")
])
print(test.messages[0].content)

print()

# Intent routing in action
for q in [
    "What does Rivian say about its market opportunity?",
    "How did RBLX perform in the first year?",
    "Score DoorDash's revenue model for clarity",
]:
    intent = classifier_chain.invoke({"query": q}).strip().upper()
    print(f"  [{intent}] {q}")

In [ ]:
from shared.lab_utils import get_scorecard
get_scorecard()

---

## Exam Preparation

### Key Concepts

| Concept | Definition |
|---|---|
| **ChatAgent** | The Databricks interface for serving agents; requires a `predict(messages, context)` method that returns `ChatAgentResponse` |
| **`predict()`** | The single required method: `predict(self, messages: list[ChatAgentMessage], context: Optional[dict] = None) -> ChatAgentResponse` |
| **ChatAgentResponse** | The response wrapper; contains a `messages` list of `ChatAgentMessage` objects with `role`, `id`, and `content` |
| **LLM-as-judge** | Using a second LLM to evaluate text quality or model output; anchored by a rubric that defines score levels |
| **Intent routing** | Classifying user intent upfront and routing to a specialized prompt/tool subset instead of letting the agent guess freely |
| **`mlflow.pyfunc.log_model`** | Logs a Python model (including ChatAgent) as an MLflow artifact; for ChatAgent, do NOT pass `signature=` explicitly |
| **`set_registry_uri("databricks-uc")`** | Redirects `mlflow.register_model()` to Unity Catalog instead of the legacy Workspace Model Registry |

### Practice Questions

**Q1.** What is the required method signature for ChatAgent?

- A) `run(self, input: str) -> str`
- B) `predict(self, messages: list[ChatAgentMessage], context: Optional[dict] = None) -> ChatAgentResponse`
- C) `invoke(self, messages: list[dict]) -> dict`
- D) `generate(self, prompt: str, **kwargs) -> str`

**Answer: B** — `predict(self, messages, context=None) -> ChatAgentResponse` is the single required method. The `messages` parameter uses `ChatAgentMessage` objects with an OpenAI-compatible schema.

---

**Q2.** Why use an LLM-as-judge instead of human evaluation at scale?

- A) LLM judges are always more accurate than human judges
- B) LLM judges are cost-effective, consistent, and reproducible across thousands of evaluations
- C) Human evaluators cannot understand financial filings
- D) LLM judges require no rubric or calibration

**Answer: B** — LLM-as-judge is cost-effective, consistent, and reproducible. Human evaluation is the gold standard for quality but doesn't scale. LLM judges still need careful rubric design and calibration.

---

**Q3.** What does `mlflow.set_registry_uri("databricks-uc")` do?

- A) Sets the MLflow tracking server URI
- B) Redirects model registration to Unity Catalog instead of the legacy Workspace Model Registry
- C) Creates a new Unity Catalog schema for models
- D) Enables experiment logging to a remote server

**Answer: B** — It redirects `mlflow.register_model()` to store model versions in Unity Catalog, which provides governance, cross-workspace access, and lineage tracking.

---

**Q4.** When should you use intent routing vs. letting the agent pick tools freely?

- A) Always — agents should never choose tools on their own
- B) When you want to reduce speculative tool calls and improve latency
- C) Only when the agent has more than 10 tools
- D) Never — intent routing adds unnecessary complexity

**Answer: B** — Intent routing reduces unnecessary tool calls and improves latency by narrowing the agent's focus to the relevant tool subset. Without it, agents may speculatively call tools to "explore" before answering.

---

**Q5.** ChatAgent auto-infers its model signature. What happens if you pass `signature=` explicitly to `mlflow.pyfunc.log_model`?

- A) It overrides the auto-inferred schema with no issues
- B) It may conflict with the auto-inferred schema, causing serving errors
- C) It is silently ignored
- D) It raises an immediate validation error

**Answer: B** — Passing `signature=` explicitly can conflict with the ChatAgent's auto-inferred schema, leading to schema mismatches during serving. The best practice is to omit it and let ChatAgent handle signature inference.

### Cost Breakdown

| Component | Detail | Est. Cost |
|---|---|---|
| Serverless compute | Notebook execution (~25 min) | ~$0.60 |
| LLM tokens | Rubric test + intent tests + agent queries | ~$1.50 |
| Vector Search | Retrieval calls during scoring and agent queries | ~$0.40 |
| MLflow logging | Model serialization and UC registration | ~$0.00 |
| **Total** | | **~$2-3** |